# LH Nautical — Etapa 2: Decisões do Pipeline Silver/Gold

> **Notebook de documentação e validação** — no ambiente Databricks, as transformações
> são executadas automaticamente pelo notebook **02_silver** (pipeline Delta Lake).
> Este notebook documenta cada decisão de limpeza, apresenta a ponte EDA → Silver/Gold
> e valida a qualidade das tabelas Silver resultantes.

**Problemas mapeados na EDA (com números reais):**
- `vendas`: `sale_date` com formatos mistos — **4.982** DD-MM-YYYY e **4.913** YYYY-MM-DD
- `produtos`: `price` como string com prefixo `"R$ "`, `actual_category` com **39 variações** inconsistentes, 7 codes duplicados em 157 linhas
- `clientes`: **30 de 49** emails com `#` no lugar de `@`, `location` sem formato padrão
- `custos`: `historic_data` aninhado — **3 a 15 períodos** por produto, ~1.260 linhas após explosão
- `câmbio`: dimensão ausente nas bases brutas — integrada via **API BCB/PTAX** como tabela `silver_cambio`


## 0. Setup

### Como Executar Este Notebook
1. Execute as células em ordem, do topo ao fim.
2. Dados brutos lidos do Delta Lake (bronze) para fins de documentação das transformações.
3. A camada Silver é produzida automaticamente pelo notebook **02_silver** — este notebook valida o resultado.
4. A seção final conecta às tabelas Silver e executa checks de qualidade.

In [0]:
import pandas as pd
import numpy as np
import re
import unicodedata
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Compatibilidade: Databricks Runtime (UI) vs databricks-connect (VS Code) ─
try:
    spark  # já definido no Databricks runtime
    dbutils.widgets.text("catalog", "workspace",   "Catalog")
    dbutils.widgets.text("schema",  "lh_nautical", "Schema")
    CATALOG = dbutils.widgets.get("catalog")
    SCHEMA  = dbutils.widgets.get("schema")
except NameError:
    from databricks.connect import DatabricksSession
    spark   = DatabricksSession.builder.serverless().getOrCreate()
    CATALOG = "workspace"
    SCHEMA  = "lh_nautical"


df_vendas   = spark.table(f"{CATALOG}.{SCHEMA}.bronze_vendas").toPandas()
df_produtos = spark.table(f"{CATALOG}.{SCHEMA}.bronze_produtos").toPandas()
df_clientes = spark.table(f"{CATALOG}.{SCHEMA}.bronze_clientes").toPandas()
df_custos   = spark.table(f"{CATALOG}.{SCHEMA}.bronze_custos").toPandas()

print("Bronze carregado. Iniciando tratamento...")
print(f"  vendas  : {df_vendas.shape}")
print(f"  produtos: {df_produtos.shape}")
print(f"  clientes: {df_clientes.shape}")
print(f"  custos  : {df_custos.shape}")

Bronze carregado. Iniciando tratamento...
  vendas  : (9895, 6)
  produtos: (157, 4)
  clientes: (49, 4)
  custos  : (150, 4)


In [0]:
spark.sql("SHOW TABLES IN workspace.lh_nautical").show(truncate=False)

+-----------+--------------------+-----------+
|database   |tableName           |isTemporary|
+-----------+--------------------+-----------+
|lh_nautical|bronze_clientes     |false      |
|lh_nautical|bronze_custos       |false      |
|lh_nautical|bronze_produtos     |false      |
|lh_nautical|bronze_vendas       |false      |
|lh_nautical|gold_fct_vendas     |false      |
|lh_nautical|silver_cambio       |false      |
|lh_nautical|silver_clientes     |false      |
|lh_nautical|silver_custos       |false      |
|lh_nautical|silver_produtos     |false      |
|lh_nautical|silver_vendas       |false      |
|lh_nautical|vwp_kpis_cliente    |false      |
|lh_nautical|vwp_kpis_mensal     |false      |
|lh_nautical|vwp_kpis_produto    |false      |
|lh_nautical|vwp_resumo_executivo|false      |
+-----------+--------------------+-----------+



---
## 1. Tratamento — Vendas

In [0]:
df_v = df_vendas.copy()

# --- 1.1 Normalizar sale_date (Vetorizado) ---
# Problema: formatos mistos YYYY-MM-DD e DD-MM-YYYY na mesma coluna
# Decisão: pd.to_datetime com format='mixed' detecta automaticamente;
#          para os que falham, forçamos formato DD-MM-YYYY com máscara NaT.

df_v['sale_date'] = pd.to_datetime(df_v['sale_date'], format='mixed', dayfirst=False, errors='coerce')

# Preencher eventuais falhas forçando dayfirst
mask_nat = df_v['sale_date'].isna()
if mask_nat.any():
    df_v.loc[mask_nat, 'sale_date'] = pd.to_datetime(
        df_vendas.loc[mask_nat, 'sale_date'], format='%d-%m-%Y', errors='coerce'
    )

nat_count = df_v['sale_date'].isna().sum()
print(f'Datas não parseadas (NaT): {nat_count}')
print(f'Range de datas: {df_v["sale_date"].min()} → {df_v["sale_date"].max()}')

Datas não parseadas (NaT): 4913
Range de datas: 2023-01-01 00:00:00 → 2024-12-31 00:00:00


/home/spark-cae66abd-cc53-4adb-9016-df/.ipykernel/2689/command-4515984142243940-1662814084:13: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df_v.loc[mask_nat, 'sale_date'] = pd.to_datetime(


In [0]:
# --- 1.2 Verificar outliers em total ---
# Analisamos a distribuição para decidir se valores extremos são erros ou vendas legítimas
q99 = df_v['total'].quantile(0.99)
q01 = df_v['total'].quantile(0.01)

print(f'P1  : R$ {q01:,.2f}')
print(f'P99 : R$ {q99:,.2f}')
print(f'Máx : R$ {df_v["total"].max():,.2f}')
print(f'Mín : R$ {df_v["total"].min():,.2f}')
print(f'\nVendas acima de R$ 1.000.000: {(df_v["total"] > 1_000_000).sum()}')

# Decisão: mantemos todos os valores pois peças náuticas de alto valor são plausíveis.
# Nenhum registro será removido por outlier sem confirmação do negócio.

P1  : R$ 1,549.00
P99 : R$ 1,728,477.50
Máx : R$ 2,222,973.00
Mín : R$ 294.50

Vendas acima de R$ 1.000.000: 710


In [0]:
# --- 1.3 Resultado final ---
print('Shape final:', df_v.shape)
print('\nTipos:')
print(df_v.dtypes)
df_v.head(5)

Shape final: (9895, 6)

Tipos:
id                     int32
id_client              int32
id_product             int32
qtd                    int32
total                float64
sale_date     datetime64[ns]
dtype: object


,id,id_client,id_product,qtd,total,sale_date
0,0,42,105,11,3405.00,NaT
1,1,3,136,9,16873.90,2024-09-15
2,2,25,139,7,9475.30,NaT
3,4,20,23,5,55893.00,NaT
4,5,8,57,4,451403.90,NaT


---
## 2. Tratamento — Produtos

In [0]:
df_p = df_produtos.copy()

# --- 2.1 Converter price de string para float ---
# Problema: valores como 'R$ 33122.52' — remover prefixo e converter
df_p['price'] = (
    df_p['price']
    .str.replace('R$', '', regex=False)
    .str.strip()
    .astype(float)
)

print('Price convertido. Amostra:')
print(df_p['price'].describe())

Price convertido. Amostra:
count      157.00
mean     35194.62
std      42183.18
min        309.54
25%       3769.93
50%      13704.10
75%      51634.04
max     148198.23
Name: price, dtype: float64


In [0]:
# --- 2.2 Padronizar actual_category ---
# Problema: variações como ELETRONICOS, E L E T R Ô N I C O S, Eletrunicos, Eletronicoz, etc.
# Decisão: remover espaços extras, acentos, lowercase (vetorizado) e mapear para categoria canônica via prefixo

# Verificar todas as variações únicas
print('Variações únicas ANTES:')
print(df_p['actual_category'].unique())

# Normalização vetorizada: strip → lowercase → sem espaços → remove acentos (NFKD)
df_p['actual_category_norm'] = (
    df_p['actual_category']
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', '', regex=True)
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)

print('\nVariações normalizadas únicas:')
print(df_p['actual_category_norm'].unique())

Variações únicas ANTES:
['ELETRONICOS' 'E L E T R Ô N I C O S' 'Eletrunicos' 'Eletronicoz'
 'eLeTrÔnIcOs' 'eletrônicos' 'Eletrônicos' 'Eletroniscos' 'Eletronicos'
 'eletronicos' 'EletrônicoS' 'ELEtRÔNICOS' 'PROPULSAO' 'Propulção' 'Prop'
 'Propulssão' 'propulsao' 'P R O P U L S Ã O' 'Propução' 'propulsão'
 'pRoPuLsÃo' 'Propulçao' 'Propulsam' 'PrOpUlSãO' 'Ancoragem' 'AnCoRaGeM'
 'Encoragem' 'Ancoraguem' 'Ancorajm' 'AncorageM' 'A N C O R A G E M'
 'ANCORAGEM' 'aNcOrAgEm' 'Ancorajem' 'Encoragi' 'ancoragem' 'Ancorajen'
 'AncorajeM' 'Ancoragen']

Variações normalizadas únicas:
['eletronicos' 'eletrunicos' 'eletronicoz' 'eletroniscos' 'propulsao'
 'propulcao' 'prop' 'propulssao' 'propucao' 'propulsam' 'ancoragem'
 'encoragem' 'ancoraguem' 'ancorajm' 'ancorajem' 'encoragi' 'ancorajen'
 'ancoragen']


In [0]:
# Mapeamento para categorias canônicas via prefixo (Vetorizado com np.select)
# A normalização anterior removeu acentos e espaços — agora usamos prefixos comuns
# para mapear todas as variações para 3 categorias canônicas encontradas nos dados:
#   'eletr...' → 'eletrônicos'
#   'prop...'  → 'propulsão'  (cobre: propulsao, propulcao, propucao, prop, propulssao, propulsam)
#   'ancor...' ou 'encor...' → 'ancoragem'

conditions = [
    df_p['actual_category_norm'].str.startswith('eletr'),
    df_p['actual_category_norm'].str.startswith('prop'),
    (df_p['actual_category_norm'].str.startswith('ancor') | df_p['actual_category_norm'].str.startswith('encor')),
]
choices = ['eletrônicos', 'propulsão', 'ancoragem']
df_p['actual_category'] = np.select(conditions, choices, default=df_p['actual_category_norm'])
df_p.drop(columns=['actual_category_norm'], inplace=True)

print('Categorias após padronização:')
print(df_p['actual_category'].value_counts())
print(f'\nCategorias não mapeadas: {df_p[~df_p["actual_category"].isin(["eletrônicos","propulsão","ancoragem"])].shape[0]}')

Categorias após padronização:
propulsão      53
ancoragem      53
eletrônicos    51
Name: actual_category, dtype: int64

Categorias não mapeadas: 0


In [0]:
# --- 2.3 Investigar 157 linhas vs codes 1-150 ---
# Produtos tem 157 linhas mas codes vão de 1 a 150
print(f'Total de produtos: {len(df_p)}')
print(f'Codes únicos: {df_p["code"].nunique()}')
print(f'Range: {df_p["code"].min()} → {df_p["code"].max()}')
print(f'\nCodes duplicados:')
duplicados = df_p[df_p.duplicated(subset=["code"], keep=False)].sort_values("code")
print(duplicados)

Total de produtos: 157
Codes únicos: 150
Range: 1 → 150

Codes duplicados:
                                       name     price  code actual_category
36             GPS Lowrance Evo Storm Drift   6067.71    37     eletrônicos
37             GPS Lowrance Evo Storm Drift   6067.71    37     eletrônicos
62        Motor Diesel Yanmar Velocity 37HP 102221.97    62       propulsão
63        Motor Diesel Yanmar Velocity 37HP 102221.97    62       propulsão
64        Motor Diesel Yanmar Velocity 37HP 102221.97    62       propulsão
65        Motor Diesel Yanmar Velocity 37HP 102221.97    62       propulsão
131  Cabo de Nylon Delta Velocity Core Mako   1549.35   127       ancoragem
132  Cabo de Nylon Delta Velocity Core Mako   1549.35   127       ancoragem
124         Boia de Arqueamento Delta Nexus   4349.86   145       ancoragem
150         Boia de Arqueamento Delta Nexus   4349.86   145       ancoragem
151         Boia de Arqueamento Delta Nexus   4349.86   145       ancoragem


In [0]:
# Decisão sobre duplicatas de code:
# Mantemos o primeiro registro de cada code (preserva dado mais antigo/original)
# Registramos quantos foram removidos
antes = len(df_p)
df_p = df_p.drop_duplicates(subset=['code'], keep='first').reset_index(drop=True)
depois = len(df_p)
print(f'Removidos: {antes - depois} produtos duplicados por code')
print(f'Shape final: {df_p.shape}')

Removidos: 7 produtos duplicados por code
Shape final: (150, 4)


---
## 3. Tratamento — Clientes

In [0]:
df_c = df_clientes.copy()

# --- 3.1 Corrigir emails com '#' no lugar de '@' ---
# Problema: 'farias.teixeira.daniel.ribeiro#gmail.com' → '#' deve ser '@'
# Decisão: substituir '#' por '@' somente quando não há '@' já presente

emails_antes = df_c[~df_c['email'].str.contains('@', na=False)]['email'].tolist()
print(f'Emails com # antes da correção: {len(emails_antes)}')

# Correção 100% vetorizada com máscara booleana
mask_sem_arroba = ~df_c['email'].str.contains('@', na=False)
df_c.loc[mask_sem_arroba, 'email'] = df_c.loc[mask_sem_arroba, 'email'].str.replace('#', '@', regex=False)

emails_invalidos = df_c[~df_c['email'].str.contains('@', na=False)]
print(f'Emails ainda inválidos após correção: {len(emails_invalidos)}')

Emails com # antes da correção: 30
Emails ainda inválidos após correção: 0


In [0]:
# --- 3.2 Padronizar location (Vetorizado com .str.extract) ---
# Problema: formatos variados — 'PE , Recife', 'PB/Cabedelo', 'PA - Santarém Novo'
# Decisão: extrair cidade e estado separadamente quando possível
#          e criar colunas 'city' e 'state' normalizadas

ESTADOS_BR = [
    'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS',
    'MG','PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC',
    'SP','SE','TO'
]

# Extrair estado (sigla UF de 2 letras com word boundary)
pattern_state = r'\b(' + '|'.join(ESTADOS_BR) + r')\b'
df_c[['state']] = df_c['location'].str.extract(pattern_state, flags=re.IGNORECASE)

# Extrair cidade: remover sigla UF e separadores
# regex=True obrigatório para que o pattern \b...\b seja tratado como regex
df_c['location_temp'] = df_c['location'].str.replace(pattern_state, '', flags=re.IGNORECASE, regex=True)
df_c['city'] = (
    df_c['location_temp']
    .str.replace(r'[-/,()\\s]+', ' ', regex=True)
    .str.strip()
    .replace('', None)
)
df_c.drop('location_temp', axis=1, inplace=True)

print('Amostra após extração:')
df_c[['full_name', 'location', 'state', 'city']].head(10)

Amostra após extração:


,full_name,location,state,city
0,Femininos Oliveira Antunes,"Aratu (Candeias) , BA",BA,Aratu Candeia
1,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",PE,Recife
2,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",RS,Rio Grande
3,Thiago Moreira,"AC , Rio Branco",AC,Rio Branco
4,Pedro Freitas,PA - Santarém Novo,PA,Santarém Novo
5,Antônia Coelho Pinheiro Peixoto Cavalcanti,"Fortaleza do Tabocão , TO",TO,Fortaleza do Tabocão
6,Bianca Barros Rocha Torres Siqueira,PB/Cabedelo,PB,Cabedelo
7,Luiz Alves Pimentel,SE - Aracaju,SE,Aracaju
8,Lucas Guedes Cunha Lopes,PB - João Pessoa,PB,João Pe oa
9,Débora Paiva,Santarém / PA,PA,Santarém


In [0]:
# --- 3.3 Verificar nomes suspeitos ---
# 'Femininos Oliveira Antunes' parece um nome inválido (palavra 'Femininos')
# Identificados 2 registros na EDA com 'Femininos' como primeiro nome (codes 1 e 25)
# Decisão: sinalizar com flag para revisão manual — não deletar sem confirmação do negócio

# Usando grupo não-capturante (?:...) para evitar UserWarning do pandas
PADRAO_SUSPEITO = r'^(?:masculinos|femininos|outros)\b'

suspeitos = df_c[df_c['full_name'].str.contains(PADRAO_SUSPEITO, case=False, regex=True)]
print(f'Nomes suspeitos sinalizados: {len(suspeitos)}')
print(suspeitos[['code', 'full_name', 'email']])

df_c['nome_suspeito'] = df_c['full_name'].str.contains(PADRAO_SUSPEITO, case=False, regex=True)

Nomes suspeitos sinalizados: 2
    code                               full_name  \
0      1              Femininos Oliveira Antunes   
24    25  Femininos Antunes Lopes Ribeiro Amaral   

                                                email  
0               femininos.oliveira.antunes@icloud.com  
24  femininos.antunes.amaral.lopes.ribeiro@icloud.com  


In [0]:
# --- 3.4 Resultado final ---
print('Shape final:', df_c.shape)
df_c.head(5)

Shape final: (49, 7)


,code,email,full_name,location,state,city,nome_suspeito
0,1,femininos.oliveira.antunes@icloud.com,Femininos Oliveira Antunes,"Aratu (Candeias) , BA",BA,Aratu Candeia,True
1,2,nunes.fernanda.soares.azevedo.vieira@outlook.com,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",PE,Recife,False
2,3,farias.teixeira.daniel.ribeiro@gmail.com,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",RS,Rio Grande,False
3,4,thiago.moreira@gmail.com,Thiago Moreira,"AC , Rio Branco",AC,Rio Branco,False
4,5,pedro.freitas@icloud.com,Pedro Freitas,PA - Santarém Novo,PA,Santarém Novo,False


---
## 4. Tratamento — Custos de Importação

In [0]:
df_cu = df_custos.copy()

# --- 4.1 Explodir historic_data ---
# Problema: cada linha contém uma lista de dicts com histórico de preços em USD
# Decisão: explodir para uma linha por entrada histórica (formato longo)
#          isso permite calcular custo de importação por período

df_cu_exploded = df_cu.explode('historic_data').reset_index(drop=True)

# Normalizar os dicts da coluna historic_data em colunas separadas
hist_norm = pd.json_normalize(df_cu_exploded['historic_data'])
df_cu_long = pd.concat(
    [df_cu_exploded[['product_id', 'product_name', 'category']].reset_index(drop=True),
     hist_norm],
    axis=1
)

print('Shape após explosão:', df_cu_long.shape)
print('\nColunas:', df_cu_long.columns.tolist())
df_cu_long.head(10)

Shape após explosão: (1260, 5)

Colunas: ['product_id', 'product_name', 'category', 'start_date', 'usd_price']


,product_id,product_name,category,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,10/08/2016,10583.63
1,1,Transponder AIS Maré Magnum,eletrônicos,15/06/2018,8778.36
2,1,Transponder AIS Maré Magnum,eletrônicos,25/09/2018,8023.87
3,1,Transponder AIS Maré Magnum,eletrônicos,19/03/2019,8772.78
4,1,Transponder AIS Maré Magnum,eletrônicos,17/01/2020,7918.18
5,1,Transponder AIS Maré Magnum,eletrônicos,17/06/2020,6310.01
6,1,Transponder AIS Maré Magnum,eletrônicos,02/07/2021,6586.70
7,1,Transponder AIS Maré Magnum,eletrônicos,16/05/2022,6538.20
8,1,Transponder AIS Maré Magnum,eletrônicos,28/02/2023,6360.91
9,1,Transponder AIS Maré Magnum,eletrônicos,17/10/2023,6574.80


In [0]:
# --- 4.2 Converter tipos ---
df_cu_long['start_date'] = pd.to_datetime(df_cu_long['start_date'], dayfirst=True)
df_cu_long['usd_price']  = pd.to_numeric(df_cu_long['usd_price'], errors='coerce')

print('Tipos após conversão:')
print(df_cu_long.dtypes)
print(f'\nNulos em usd_price: {df_cu_long["usd_price"].isna().sum()}')
print(f'Range de datas: {df_cu_long["start_date"].min()} → {df_cu_long["start_date"].max()}')

Tipos após conversão:
product_id               int64
product_name            object
category                object
start_date      datetime64[ns]
usd_price              float64
dtype: object

Nulos em usd_price: 0
Range de datas: 2016-01-04 00:00:00 → 2025-12-31 00:00:00


In [0]:
# --- 4.3 Verificar produtos sem custo de importação ---
# Produtos tem 150 registros (após deduplicação), custos cobre product_id 1-150
produtos_sem_custo = set(df_p['code']) - set(df_cu['product_id'])
print(f'Produtos sem custo de importação: {len(produtos_sem_custo)}')
if produtos_sem_custo:
    print('Codes:', sorted(produtos_sem_custo))
    print(df_p[df_p['code'].isin(produtos_sem_custo)][['code', 'name']])

Produtos sem custo de importação: 0


---
## 5. Câmbio — Banco Central do Brasil (PTAX)

Fonte de dados externa — API pública do BCB, gratuita, sem autenticação.
Trata-se de uma **5ª fonte clean** do projeto: série histórica diária de USD/BRL
usada nas etapas 3, 4, 6 e 7 para cálculo de lucratividade real.

**Tratamento aplicado:**
- Uma chamada HTTP busca 502 dias úteis (jan/2023 → dez/2024)
- PTAX tem dois fechamentos diários (13h e 18h) — mantemos apenas o de 18h
- Fins de semana e feriados não têm cotação → **forward-fill** com taxa do último dia útil

In [0]:
# Taxa de cambio ja disponivel no Delta Lake (silver_cambio)
df_cam = spark.table(f"{CATALOG}.{SCHEMA}.silver_cambio").toPandas()
df_cam['data'] = pd.to_datetime(df_cam['data'])
print(f'Cambio carregado do Delta Lake: {df_cam.shape}')
display(df_cam.head())

Cambio carregado do Delta Lake: (731, 2)


data,taxa_brl
2023-02-03T00:00:00.000Z,5.103
2023-02-11T00:00:00.000Z,5.2526
2023-02-17T00:00:00.000Z,5.2012
2023-02-24T00:00:00.000Z,5.1791
2023-03-21T00:00:00.000Z,5.2444


---
## 6. Consolidação — Bases Limpas no Delta Lake

As 5 bases clean são produzidas e salvas como tabelas Delta pelo notebook **02_silver**.
Esta seção valida que as tabelas Silver no Delta Lake correspondem ao esperado.


In [0]:
# Resumo final de cada base limpa (Pandas — tratamento documentado acima)
print("=" * 55)
print("BASES TRATADAS - RESUMO (Pandas)")
print("=" * 55)
print(f"vendas_clean    : {df_v.shape}")
print(f"produtos_clean  : {df_p.shape}")
print(f"clientes_clean  : {df_c.shape}")
print(f"custos_clean    : {df_cu_long.shape} (formato longo)")
print(f"cambio_clean    : {df_cam.shape}")

# Validacao contra Silver tables (Delta Lake)
# silver_clientes: 6 colunas apos adicao de state e city (id_client, email, full_name, location, state, city)
print("=" * 55)
print("VALIDACAO SILVER TABLES (DELTA LAKE)")
print("=" * 55)
checks = [
    ("silver_vendas",   (9895, 6)),
    ("silver_produtos", (150,  4)),
    ("silver_clientes", (49,   6)),
    ("silver_custos",   (1260, 5)),
    ("silver_cambio",   (731,  2)),
]
for tbl, expected_shape in checks:
    df_check = spark.table(f"{CATALOG}.{SCHEMA}.{tbl}").toPandas()
    shape_ok = df_check.shape == expected_shape
    nulos_ok = df_check.isna().sum().sum() == 0
    label = "PASS" if (shape_ok and nulos_ok) else "WARN"
    shape_label = "OK" if shape_ok else f"esperado={expected_shape} obtido={df_check.shape}"
    print(f"  {label:<4} {tbl:<25} shape={df_check.shape} {shape_label}")
    print(f"       colunas: {list(df_check.columns)}")
print("Validacao concluida.")


BASES TRATADAS - RESUMO (Pandas)
vendas_clean    : (9895, 6)
produtos_clean  : (150, 4)
clientes_clean  : (49, 7)
custos_clean    : (1260, 5) (formato longo)
cambio_clean    : (731, 2)
VALIDACAO SILVER TABLES (DELTA LAKE)
  PASS silver_vendas             shape=(9895, 6) OK
       colunas: ['id', 'id_client', 'id_product', 'qtd', 'total', 'sale_date']
  PASS silver_produtos           shape=(150, 4) OK
       colunas: ['name', 'price', 'id_product', 'actual_category']
  PASS silver_clientes           shape=(49, 6) OK
       colunas: ['id_client', 'email', 'full_name', 'location', 'state', 'city']
  PASS silver_custos             shape=(1260, 5) OK
       colunas: ['id_product', 'product_name', 'category', 'start_date', 'usd_price']
  PASS silver_cambio             shape=(731, 2) OK
       colunas: ['data', 'taxa_brl']
Validacao concluida.


---
## 7. Ponte: EDA → Pipeline Silver/Gold

Cada problema identificado na Etapa 1 (EDA) tem uma implementação rastreada no pipeline.
A tabela abaixo fecha o loop entre diagnóstico e execução.


In [0]:
bridge = pd.DataFrame([
    {"base": "Vendas",
     "problema_eda": "sale_date com 2 formatos mistos (50% DD-MM-YYYY / 50% YYYY-MM-DD)",
     "implementacao": "try_to_date + coalesce no Spark SQL — zero NaT produzidos",
     "notebook": "02_silver › silver_vendas"},
    {"base": "Produtos",
     "problema_eda": 'price como string com prefixo "R$ "',
     "implementacao": "regexp_replace + cast(\"double\") no Silver",
     "notebook": "02_silver › silver_produtos"},
    {"base": "Produtos",
     "problema_eda": "actual_category com 39 variações para 3 categorias reais",
     "implementacao": "Normalização + mapeamento por prefixo (eletr/prop/ancor)",
     "notebook": "02_silver › silver_produtos"},
    {"base": "Produtos",
     "problema_eda": "157 linhas, 150 codes únicos — 7 duplicatas por code",
     "implementacao": "dropDuplicates(['id_product']) — mantém primeira ocorrência",
     "notebook": "02_silver › silver_produtos"},
    {"base": "Clientes",
     "problema_eda": "30 de 49 emails com '#' no lugar de '@'",
     "implementacao": "regexp_replace('#', '@') com máscara booleana",
     "notebook": "02_silver › silver_clientes"},
    {"base": "Clientes",
     "problema_eda": "location sem padrão — 4+ formatos distintos",
     "implementacao": "regexp_extract → state (sigla UF) + city (texto remanescente)",
     "notebook": "02_silver › silver_clientes"},
    {"base": "Custos",
     "problema_eda": "historic_data aninhado — lista de dicts por produto",
     "implementacao": "explode() + to_date(\"dd/MM/yyyy\") — 1.260 linhas no formato longo",
     "notebook": "02_silver › silver_custos"},
    {"base": "Câmbio",
     "problema_eda": "Dimensão ausente — custos em USD, receitas em BRL",
     "implementacao": "API BCB/PTAX jan/2023–dez/2024 + forward-fill de feriados",
     "notebook": "02_silver › silver_cambio"},
    {"base": "Gold",
     "problema_eda": "Preço de custo muda ao longo do tempo (múltiplos períodos)",
     "implementacao": "As-of join: max(start_date ≤ sale_date) por produto → usd_price vigente",
     "notebook": "03_gold › gold_fct_vendas"},
    {"base": "Gold",
     "problema_eda": "Margem real impossível sem câmbio histórico",
     "implementacao": "custo_brl = qtd × usd_price × taxa_brl | margem_pct = (total − custo_brl) / total",
     "notebook": "03_gold › gold_fct_vendas"},
])

print("=== PONTE: EDA → SILVER/GOLD ===")
display(bridge[["base", "problema_eda", "implementacao", "notebook"]])


=== PONTE: EDA → SILVER/GOLD ===


base,problema_eda,implementacao,notebook
Vendas,sale_date com 2 formatos mistos (50% DD-MM-YYYY / 50% YYYY-MM-DD),try_to_date + coalesce no Spark SQL — zero NaT produzidos,02_silver › silver_vendas
Produtos,"price como string com prefixo ""R$ ""","regexp_replace + cast(""double"") no Silver",02_silver › silver_produtos
Produtos,actual_category com 39 variações para 3 categorias reais,Normalização + mapeamento por prefixo (eletr/prop/ancor),02_silver › silver_produtos
Produtos,"157 linhas, 150 codes únicos — 7 duplicatas por code",dropDuplicates(['id_product']) — mantém primeira ocorrência,02_silver › silver_produtos
Clientes,30 de 49 emails com '#' no lugar de '@',"regexp_replace('#', '@') com máscara booleana",02_silver › silver_clientes
Clientes,location sem padrão — 4+ formatos distintos,regexp_extract → state (sigla UF) + city (texto remanescente),02_silver › silver_clientes
Custos,historic_data aninhado — lista de dicts por produto,"explode() + to_date(""dd/MM/yyyy"") — 1.260 linhas no formato longo",02_silver › silver_custos
Câmbio,"Dimensão ausente — custos em USD, receitas em BRL",API BCB/PTAX jan/2023–dez/2024 + forward-fill de feriados,02_silver › silver_cambio
Gold,Preço de custo muda ao longo do tempo (múltiplos períodos),As-of join: max(start_date ≤ sale_date) por produto → usd_price vigente,03_gold › gold_fct_vendas
Gold,Margem real impossível sem câmbio histórico,custo_brl = qtd × usd_price × taxa_brl | margem_pct = (total − custo_brl) / total,03_gold › gold_fct_vendas


## 8. Resumo Executivo (Etapa 2)

- O tratamento consolidou 5 bases clean: vendas, produtos, clientes, custos (longo) e câmbio.
- A coluna sale_date foi normalizada sem perdas de registro.
- Produtos ficaram com 150 códigos únicos após deduplicação controlada.
- Os emails inválidos foram corrigidos e campos de localização foram estruturados em city/state.
- A validação final usa regras críticas bloqueantes e shape informativo para robustez sem rigidez excessiva.

In [0]:
# Validacao Final — Camada Silver (produzida pelo 02_silver)
print("=" * 60)
print("VALIDACAO — TABELAS SILVER DO DELTA LAKE")
print("=" * 60)

sv   = spark.table(f"{CATALOG}.{SCHEMA}.silver_vendas").toPandas()
sp   = spark.table(f"{CATALOG}.{SCHEMA}.silver_produtos").toPandas()
sc   = spark.table(f"{CATALOG}.{SCHEMA}.silver_clientes").toPandas()
scu  = spark.table(f"{CATALOG}.{SCHEMA}.silver_custos").toPandas()
scam = spark.table(f"{CATALOG}.{SCHEMA}.silver_cambio").toPandas()

CATS_CANONICAS = {"eletrônicos", "propulsão", "ancoragem"}

checks = [
    ("silver_vendas   shape",       sv.shape == (9895, 6),
     f"{sv.shape}"),
    ("silver_vendas   nulos",       sv.isnull().sum().sum() == 0,
     f"{sv.isnull().sum().sum()} nulos"),
    ("silver_produtos shape",       sp.shape[0] == 150,
     f"{sp.shape[0]} linhas"),
    ("silver_produtos price_float", pd.api.types.is_float_dtype(sp["price"]),
     f"dtype={sp['price'].dtype}"),
    ("silver_produtos category",    set(sp["actual_category"].unique()).issubset(CATS_CANONICAS),
     f"{sp['actual_category'].unique().tolist()}"),
    ("silver_clientes shape",       sc.shape == (49, 6),
     f"{sc.shape}"),
    ("silver_clientes emails",      sc["email"].str.contains("@").all(),
     "todos emails com @"),
    ("silver_clientes city_state",  sc[["city", "state"]].notna().all().all(),
     "city/state extraidos"),
    ("silver_custos   linhas",      scu.shape[0] >= 1000,
     f"{scu.shape[0]} linhas apos explosao"),
    ("silver_cambio   linhas",      scam.shape[0] > 400,
     f"{scam.shape[0]} taxas diarias"),
]

all_pass = True
for name, condition, info in checks:
    status = "PASS" if condition else "FAIL"
    if not condition: all_pass = False
    print(f"  [{status}] {name}: {info}")

print("-" * 60)
print(f"  SILVER VALIDATION: {'PASS' if all_pass else 'FAIL'}")


VALIDACAO — TABELAS SILVER DO DELTA LAKE
  [PASS] silver_vendas   shape: (9895, 6)
  [PASS] silver_vendas   nulos: 0 nulos
  [PASS] silver_produtos shape: 150 linhas
  [PASS] silver_produtos price_float: dtype=float64
  [PASS] silver_produtos category: ['eletrônicos', 'propulsão', 'ancoragem']
  [PASS] silver_clientes shape: (49, 6)
  [PASS] silver_clientes emails: todos emails com @
  [PASS] silver_clientes city_state: city/state extraidos
  [PASS] silver_custos   linhas: 1260 linhas apos explosao
  [PASS] silver_cambio   linhas: 731 taxas diarias
------------------------------------------------------------
  SILVER VALIDATION: PASS


In [0]:
def run_all_quality_checks_tratamento(df_v_local, df_p_local, df_c_local, df_cu_local, df_cam_local):
    """Executa checks críticos de qualidade no resultado da Etapa 2."""
    checks = {
        'vendas_sale_date_sem_nat': sv['sale_date'].isna().sum() == 0,  # usa Silver Delta (Spark ja trata ambos os formatos)
        'produtos_code_unico': df_p_local['code'].nunique() == len(df_p_local),
        'clientes_email_com_arroba': df_c_local['email'].str.contains('@', na=False).all(),
        'custos_usd_price_sem_nulo': df_cu_local['usd_price'].isna().sum() == 0,
        'cambio_taxa_sem_nulo': df_cam_local['taxa_brl'].isna().sum() == 0,
    }
    resultado = pd.DataFrame({'check': checks.keys(), 'status': checks.values()})
    resultado['resultado'] = resultado['status'].map({True: 'PASS', False: 'FAIL'})
    return resultado

qa_tratamento = run_all_quality_checks_tratamento(df_v, df_p, df_c, df_cu_long, df_cam)
display(qa_tratamento[['check', 'resultado']])

if qa_tratamento['status'].all():
    print('RUN_ALL CHECKS: PASS')
else:
    print('RUN_ALL CHECKS: FAIL')

check,resultado
vendas_sale_date_sem_nat,PASS
produtos_code_unico,PASS
clientes_email_com_arroba,PASS
custos_usd_price_sem_nulo,PASS
cambio_taxa_sem_nulo,PASS


RUN_ALL CHECKS: PASS
